### Model Training Pipeline
Feature Engineering → Selection → SMOTE → XGBoost → Threshold Tuning

Setup & Load Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve, f1_score,
                              precision_score, recall_score, accuracy_score)
from imblearn.over_sampling import SMOTE

In [3]:
df = pd.read_csv("../outputs/outliers_handled.csv")
print("Loaded shape:", df.shape)
print("Columns:", df.columns.tolist())
print(f"\nTarget: {df['Target'].value_counts().to_dict()}")

Loaded shape: (149233, 11)
Columns: ['Target', 'RevolvingUtilization', 'Age', 'Times30_59Late', 'DebtRatio', 'MonthlyIncome', 'OpenCreditLines', 'Times90Late', 'RealEstateLines', 'Times60_89Late', 'Dependents']

Target: {0: 139229, 1: 10004}


Feature Engineering

Adding 3 new features that combine weak signals into stronger ones.

In [4]:
# 1. Total late payments — combining 3 separate late payment columns into one
df["TotalLatePayments"] = df["Times30_59Late"] + df["Times60_89Late"] + df["Times90Late"]

# 2. Has severe delinquency 
df["HasSevereDelinquency"] = (df["Times90Late"] > 0).astype(int)

# 3. Income to debt ratio 
df["IncomeDebtRatio"] = df["MonthlyIncome"] / (df["DebtRatio"] + 1)

print("Engineered features added:")
print(f"  TotalLatePayments     : min={df['TotalLatePayments'].min()}, max={df['TotalLatePayments'].max()}")
print(f"  HasSevereDelinquency  : {df['HasSevereDelinquency'].value_counts().to_dict()}")
print(f"  IncomeDebtRatio       : median={df['IncomeDebtRatio'].median():.1f}")

# Capping IncomeDebtRatio at 99th percentile 
cap = df["IncomeDebtRatio"].quantile(0.99)
df["IncomeDebtRatio"] = df["IncomeDebtRatio"].clip(upper=cap)

print(f"\nShape after engineering: {df.shape}")
print(f"All features: {[c for c in df.columns if c != 'Target']}")

Engineered features added:
  TotalLatePayments     : min=0, max=60
  HasSevereDelinquency  : {0: 140961, 1: 8272}
  IncomeDebtRatio       : median=3370.2

Shape after engineering: (149233, 14)
All features: ['RevolvingUtilization', 'Age', 'Times30_59Late', 'DebtRatio', 'MonthlyIncome', 'OpenCreditLines', 'Times90Late', 'RealEstateLines', 'Times60_89Late', 'Dependents', 'TotalLatePayments', 'HasSevereDelinquency', 'IncomeDebtRatio']


### Validation Strategy Justification

We use a **70/15/15 stratified Train/Validation/Test split** combined with **5-Fold Stratified Cross-Validation**:

- **Training set (70%)**: Used to train the model. SMOTE is applied only here.
- **Validation set (15%)**: Used for model development — feature selection, threshold tuning, and comparing approaches. We iterate on this set multiple times.
- **Test set (15%)**: Touched only ONCE at the very end for final unbiased evaluation. This simulates true unseen data.
- **Stratified splitting** ensures all three sets maintain the original 93/7 class ratio.
- **5-Fold CV** is run separately to confirm consistency across different data splits.

In [5]:
feature_cols = [col for col in df.columns if col != "Target"]
X = df[feature_cols]
y = df["Target"]

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: split the 30% into 15% validation + 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Training set  : {X_train.shape[0]:,} rows (70%)")
print(f"Validation set: {X_val.shape[0]:,} rows (15%)")
print(f"Test set      : {X_test.shape[0]:,} rows (15%)")
print(f"Features      : {len(feature_cols)}")

print(f"\nTarget distribution per split:")
for name, ys in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"  {name:<6}: No Default {(ys==0).sum():>7,} ({(ys==0).mean()*100:.1f}%) | Default {(ys==1).sum():>5,} ({(ys==1).mean()*100:.1f}%)")

Training set  : 104,463 rows (70%)
Validation set: 22,385 rows (15%)
Test set      : 22,385 rows (15%)
Features      : 13

Target distribution per split:
  Train : No Default  97,460 (93.3%) | Default 7,003 (6.7%)
  Val   : No Default  20,884 (93.3%) | Default 1,501 (6.7%)
  Test  : No Default  20,885 (93.3%) | Default 1,500 (6.7%)


Scaling (fit on train only)

In [ ]:
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=feature_cols, index=X_val.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols, index=X_test.index)
print("RobustScaler fitted on training set, applied to train/val/test")